# Who Won

**Analytical purpose:** How did USA and USSR medal totals compare edition by edition?

This notebook is the official chart-specific preprocessing pipeline. Shared Olympic/geography logic lives in `common.py`.

In [ ]:
from pathlib import Path
import sys

CHARTS_DIR = Path.cwd()
if CHARTS_DIR.name != 'charts':
    candidates = [p / 'preprocessing' / 'charts' for p in [Path.cwd(), *Path.cwd().parents]]
    CHARTS_DIR = next((p for p in candidates if (p / 'common.py').exists()), None)
    if CHARTS_DIR is None:
        raise RuntimeError('Run this notebook from the repository or preprocessing/charts directory.')
sys.path.insert(0, str(CHARTS_DIR))
from common import *
ensure_output_dirs()

In [ ]:
import pandas as pd
import numpy as np

common = load_common()
rows = []
for year in RIVALRY_YEARS:
    for noc in ["USA", "URS"]:
        match = common[(common.Year == year) & (common.NOC == noc)]
        if match.empty:
            rows.append({
                "Year": year, "City": "", "NOC": noc, "Country": SUPERPOWER_NAMES[noc],
                "TotalMedals": np.nan, "GoldMedals": np.nan,
                "ParticipationStatus": "boycott" if BOYCOTTS.get(year) == noc else "did_not_participate",
                "BoycottBy": BOYCOTTS.get(year, ""),
            })
        else:
            row = match.iloc[0]
            rows.append({
                "Year": year, "City": row.City, "NOC": noc, "Country": row.Country,
                "TotalMedals": row.TotalMedals, "GoldMedals": row.GoldMedals,
                "ParticipationStatus": row.ParticipationStatus, "BoycottBy": row.BoycottBy,
            })

out = pd.DataFrame(rows)
assert len(out) == 20
path = FINAL_DIR / "who_won.csv"
out.to_csv(path, index=False)
print(f"Wrote {path.relative_to(REPO_ROOT)}: {len(out)} rows")
out.head()